# Tome C1, Chapter 4: Generating Functions and the Symbolic Method (Python companion)

Exercises first, solutions at the end. We turn the symbolic dictionary into a small library: describe a class, get its counting sequence. Then we count words accepted by automata, settle Penney's game with Conway's formula and check it by simulation, count labelled structures, implement Lagrange inversion and test it on random equations, and measure how deep random trees are.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, time
from fractions import Fraction
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
random.seed(4)
N = 30          # series are truncated after z^N

## Exercises

**P4.1 (★★) A symbolic engine.** Represent a series by the list of its first $N+1$ coefficients. Implement the constructions of Theorem 4.1 (union, product, SEQ, MSET, PSET, CYC) and use them to count compositions, partitions, partitions into distinct and into odd parts (Exercise 4.3), binary necklaces (compare with brute force), plane trees and unlabelled rooted trees, solving recursive specifications by iteration.

**P4.2 (★) Automata.** Write a function that counts the words of each length accepted by an automaton with the transition matrix (Theorem 4.2). Check the words without $11$ (Figure 4.2) and the multiples of $3$ (Exercise 4.4) against brute force, and let sympy find the rational generating function.

**P4.3 (★★) Penney's game.** Implement Conway numbers and the formulas of Corollary 4.4. Check them against brute force (words avoiding a pattern) and against simulation (waiting times, who wins). Print the best reply to each pattern of length $3$, then of length $4$. How long does a monkey need to type ABRACADABRA?

**P4.4 (★★) Labelled structures.** With exponential generating functions in exact rational arithmetic, compute the Fubini numbers (Exercise 4.7) and compare with $\frac{n!}{2(\ln 2)^{n+1}}$, the Bell numbers, the Cayley trees from $T=ze^T$ (by iteration), and check that $\frac1{1-T}$ counts the $n^n$ maps. Count rooted forests by brute force for small $n$ (Exercise 4.10).

**P4.5 (★★) Lagrange inversion.** Implement $[z^n]T^k=\frac kn[w^{n-k}]\varphi^n$ and compare with the solution of $T=z\varphi(T)$ computed by iteration, for random $\varphi$ with rational coefficients of both signs (the case the cycle-lemma proof reaches only through the polynomial argument). Then compute Motzkin and Narayana numbers (Exercises 4.11 and 4.12).

**P4.6 (★★★) How deep is a random tree?** (a) Check Theorem 4.9 by enumerating all plane trees with $n\le13$ vertices. (b) Sample random plane trees with up to $10^5$ vertices with the cycle lemma, and compare their average depth with $\frac12\sqrt{\pi n}$ and their height with $\sqrt{\pi n}$. (c) Compare with random binary search trees. (d) Check the mean and variance of the number of cycles of a random permutation (Example 4.8, Exercise 4.8).

## Solutions

### P4.1
**How to find it.** Products are truncated convolutions; $\frac1{1-A}$ is solved coefficient by coefficient; the exponential forms of MSET and PSET need $A(z^k)$, which is a reindexing; CYC needs $\log\frac1{1-B}=\sum_mB^m/m$. A recursive class is the limit of the iteration $\mathcal G\leftarrow\Phi(\mathcal G)$ started at $0$: each round fixes at least one more coefficient.

In [ ]:
def mul(a, b):
    c = [0] * (N + 1)
    for i, x in enumerate(a):
        if x:
            for j in range(N + 1 - i):
                c[i + j] += x * b[j]
    return c
def add(*xs): return [sum(t) for t in zip(*xs)]
def scal(c, a): return [c * x for x in a]
def Z(): return [0, 1] + [0] * (N - 1)
def ONE(): return [1] + [0] * N
def SEQ(a):
    assert a[0] == 0
    b = [0] * (N + 1); b[0] = 1
    for n in range(1, N + 1):
        b[n] = sum(a[k] * b[n - k] for k in range(1, n + 1))
    return b
def subs_pow(a, k):
    c = [0] * (N + 1)
    for i, x in enumerate(a):
        if i * k <= N: c[i * k] += x
    return c
def exp_series(a):
    # exp of a series with a[0] = 0, exact in Fractions, via b' = a' b
    assert a[0] == 0
    b = [Fraction(0)] * (N + 1); b[0] = Fraction(1)
    for n in range(1, N + 1):
        b[n] = sum(Fraction(k) * a[k] * b[n - k] for k in range(1, n + 1)) / n
    return b
def MSET(a): return [int(x) for x in exp_series([sum(Fraction(subs_pow(a, k)[n], k) for k in range(1, N + 1)) for n in range(N + 1)])]
def PSET(a): return [int(x) for x in exp_series([sum(Fraction((-1) ** (k - 1) * subs_pow(a, k)[n], k) for k in range(1, N + 1)) for n in range(N + 1)])]
def log_inv(b):   # log 1/(1-b) = sum b^m / m
    out = [Fraction(0)] * (N + 1); p = ONE()
    for m in range(1, N + 1):
        p = mul(p, b)
        out = [o + Fraction(x, m) for o, x in zip(out, p)]
    return out
def CYC(a):
    tot = [Fraction(0)] * (N + 1)
    for d in range(1, N + 1):
        L = log_inv(subs_pow(a, d))
        tot = [t + Fraction(sp.totient(d), d) * x for t, x in zip(tot, L)]
    return [int(x) for x in tot]
def fix(Phi):
    g = [0] * (N + 1)
    for _ in range(N + 1): g = Phi(g)
    return g

SEQ1 = SEQ(Z())[:]; SEQ1[0] = 0                     # SEQ_{>=1}(Z) = z/(1-z)
print('compositions        ', SEQ(SEQ1)[:12])
print('partitions          ', MSET(SEQ1)[:12])
print('distinct parts      ', PSET(SEQ1)[:12])
odd = [1 if n % 2 else 0 for n in range(N + 1)]
print('odd parts           ', MSET(odd)[:12])
neck = CYC(scal(2, Z()))
brute = [len({min(w[i:] + w[:i] for i in range(n)) for w in itertools.product('01', repeat=n)}) for n in range(1, 13)]
print('binary necklaces    ', neck[1:13]); print('  brute force       ', brute)
G = fix(lambda g: mul(Z(), SEQ(g)))
print('plane trees         ', G[:12])
T = fix(lambda t: mul(Z(), MSET(t)))
print('rooted trees        ', T[:12])
assert PSET(SEQ1) == MSET(odd) and neck[1:13] == brute

Twenty lines of code count partitions, necklaces and trees. The equality of the distinct-part and odd-part counts is Euler's theorem of Exercise 4.3; a recursive specification, iterated, converges one coefficient per round.

### P4.2
**How to find it.** The vector $v_n$ whose $s$-th entry is the number of words of length $n$ leading from $s$ to an accepting state satisfies $v_{n+1}=Mv_n$, $v_0=\mathbf 1_F$ (the proof of Theorem 4.2, coefficient by coefficient). Sympy inverts $I-zM$ symbolically.

In [ ]:
def count_accepted(M, start, accept, n_max):
    v = np.array([1 if s in accept else 0 for s in range(len(M))], dtype=object)
    M = np.array(M, dtype=object); out = []
    for n in range(n_max + 1):
        out.append(v[start]); v = M.dot(v)
    return out

# no two consecutive 1s: states a=0, b=1, sink=2
M11 = [[1, 1, 0], [1, 0, 1], [0, 0, 2]]
print('no 11 :', count_accepted(M11, 0, {0, 1}, 12))
print('brute :', [sum('11' not in ''.join(w) for w in itertools.product('01', repeat=n)) for n in range(13)])
# multiples of 3
M3 = [[0] * 3 for _ in range(3)]
for r in range(3):
    for b in range(2): M3[r][(2 * r + b) % 3] += 1
print('mod 3 :', count_accepted(M3, 0, {0}, 12))
print('brute :', [sum(int(''.join(w) or '0', 2) % 3 == 0 for w in itertools.product('01', repeat=n)) for n in range(13)])
z = sp.symbols('z')
L = (sp.eye(3) - z * sp.Matrix(M3)).inv() * sp.Matrix([1, 0, 0])
print('L_0 =', sp.factor(sp.simplify(L[0])), ' =', sp.apart(sp.simplify(L[0]), z))

### P4.3
**How to find it.** The Conway number $XY$ sums $q^j$ over the overlaps $j$ (suffix of $X$ of length $j$ equal to the prefix of $Y$). Then the waiting time is $AA$, and $A$ beats $B$ with odds $(BB-BA):(AA-AB)$. For the counting formula, expand $c(z)/(z^k+(1-qz)c(z))$ as a power series.

In [ ]:
def conway(X, Y, q=2):
    k = len(X)
    return sum(q ** j for j in range(1, k + 1) if X[-j:] == Y[:j])

def avoid_counts(A, alphabet, n_max):
    q, k = len(alphabet), len(A)
    zz = sp.symbols('zz')
    c = sum(zz ** (k - j) for j in range(1, k + 1) if A[-j:] == A[:j])
    ser = sp.series(c / (zz ** k + (1 - q * zz) * c), zz, 0, n_max + 1).removeO()
    return [int(ser.coeff(zz, n)) for n in range(n_max + 1)]

for A in ['HH', 'HT', 'HHT', 'HTH', 'HHHH']:
    f = avoid_counts(A, 'HT', 10)
    b = [sum(A not in ''.join(w) for w in itertools.product('HT', repeat=n)) for n in range(11)]
    assert f == b
    print(f'{A:5} avoiding counts {f}   expected waiting time {conway(A, A)}')

def play(A, B, rng):
    s = ''
    while True:
        s += rng.choice('HT')
        if s.endswith(A): return 'A', len(s)
        if s.endswith(B): return 'B', len(s)
rng = random.Random(1)
for A, B in [('THH', 'HHT'), ('HH', 'HT'), ('THH', 'HHH')]:
    res = [play(A, B, rng) for _ in range(40000)]
    pA = sum(r == 'A' for r, _ in res) / len(res)
    num, den = conway(B, B) - conway(B, A), conway(A, A) - conway(A, B)
    print(f'{A} vs {B}: simulated P(A first) = {pA:.4f}, Conway {num}:{den} -> {num / (num + den):.4f}')
for A in ['HH', 'HT']:
    w = [play(A, 'X' * 9, rng)[1] for _ in range(40000)]
    print(f'waiting time for {A}: simulated {np.mean(w):.3f}, formula {conway(A, A)}')

In [ ]:
def best_reply(B):
    pats = [''.join(p) for p in itertools.product('HT', repeat=len(B))]
    best = max((p for p in pats if p != B), key=lambda A: Fraction(conway(B, B) - conway(B, A), conway(A, A) - conway(A, B)))
    o = Fraction(conway(B, B) - conway(B, best), conway(best, best) - conway(best, B))
    return best, o
for k in (3, 4):
    print(f'length {k}:')
    for B in (''.join(p) for p in itertools.product('HT', repeat=k)):
        A, o = best_reply(B)
        rule = ('T' if B[1] == 'H' else 'H') + B[:-1]
        print(f'   against {B} play {A} (odds {o.numerator}:{o.denominator})' + ('' if A == rule else f'   [rule gives {rule}: odds {Fraction(conway(B, B) - conway(B, rule), conway(rule, rule) - conway(rule, B))}]'))
W = 'ABRACADABRA'
print('ABRACADABRA: overlaps', [j for j in range(1, 12) if W[-j:] == W[:j]], ' expected letters', conway(W, W, 26))

For length 3 the rule $\bar b_2b_1b_2$ is always optimal. For length 4 it still always wins, though not always with the best odds: Conway's formula is the referee that finds the best reply.

### P4.4
**How to find it.** Keep EGFs as lists of Fractions $a_n/n!$. The product of EGFs is the product of the lists; $\frac1{1-A}$ and $\exp A$ are computed as in P4.1. For forests, a rooted forest on $\{1,\dots,n\}$ is a parent function with no cycle (roots have no parent).

In [ ]:
def egf_to_counts(a): return [int(x * math.factorial(n)) for n, x in enumerate(a)]
ez1 = [Fraction(0)] + [Fraction(1, math.factorial(n)) for n in range(1, N + 1)]   # e^z - 1
fub = SEQ(ez1)
print('Fubini :', egf_to_counts(fub)[:10])
for n in [5, 10, 20, 30]:
    print(f'   n={n:2}: f_n / (n!/(2 ln2^(n+1))) = {egf_to_counts(fub)[n] / (math.factorial(n) / (2 * math.log(2) ** (n + 1))):.12f}')
print('Bell   :', egf_to_counts(exp_series(ez1))[:10])
T = [Fraction(0)] * (N + 1)
for _ in range(N + 1):
    T = mul(Z(), exp_series(T))
t = egf_to_counts(T)
print('Cayley trees:', t[1:9], ' n^(n-1):', [n ** (n - 1) for n in range(1, 9)])
maps = egf_to_counts(SEQ(T))
print('maps        :', maps[:9], ' n^n:', [n ** n for n in range(9)])

def forests_brute(n):
    cnt = Counter()
    for parent in itertools.product(range(-1, n), repeat=n):
        if any(parent[i] == i for i in range(n)): continue
        ok = True
        for i in range(n):           # follow parents: must reach a root within n steps
            x, steps = i, 0
            while parent[x] != -1 and steps <= n: x = parent[x]; steps += 1
            if parent[x] != -1: ok = False; break
        if ok: cnt[sum(p == -1 for p in parent)] += 1
    return cnt
for n in range(1, 7):
    b = forests_brute(n)
    print(n, [b[k] for k in range(1, n + 1)], [math.comb(n - 1, k - 1) * n ** (n - k) for k in range(1, n + 1)])

### P4.5
**How to find it.** $[w^{n-k}]\varphi^n$ is a coefficient of a polynomial power: truncate $\varphi$ to degree $n$ and multiply. On the other side, iterate $T\leftarrow z\varphi(T)$ with exact rational coefficients.

In [ ]:
def compose(phi, t):   # phi(t) for t with t[0] = 0
    res = [Fraction(0)] * (N + 1); p = [Fraction(1)] + [Fraction(0)] * N
    for c in phi:
        res = [r + c * x for r, x in zip(res, p)]
        p = mul(p, t)
    return res
def solve(phi):
    t = [Fraction(0)] * (N + 1)
    for _ in range(N + 1): t = mul(Z(), compose(phi, t))
    return t
def lagrange(phi, n, k):
    p = [Fraction(1)] + [Fraction(0)] * N
    for _ in range(n): p = mul(p, list(phi) + [Fraction(0)] * (N + 1 - len(phi)))
    return Fraction(k, n) * p[n - k]

rng = random.Random(7)
for trial in range(5):
    phi = [Fraction(rng.randint(-5, 5), rng.randint(1, 4)) for _ in range(5)]
    if phi[0] == 0: phi[0] = Fraction(1)
    t = solve(phi); tk = [Fraction(1)] + [Fraction(0)] * N
    ok = True
    for k in range(1, 5):
        tk = mul(tk, t)
        ok &= all(tk[n] == lagrange(phi, n, k) for n in range(k, 13))
    print('phi =', [str(c) for c in phi], ' Lagrange agrees for n <= 12, k <= 4:', ok)
print('Motzkin :', [int(lagrange([1, 1, 1], n, 1)) for n in range(1, 13)])
u = sp.symbols('u')
for n in range(2, 8):
    w_ = sp.symbols('w_')
    poly = sp.expand(sp.series((u + w_ / (1 - w_)) ** n, w_, 0, n).removeO())
    coeffs = sp.Poly(sp.expand(poly.coeff(w_, n - 1) / n), u).all_coeffs()[::-1]
    print(f'n={n}: leaves distribution {coeffs[1:]}  Narayana {[math.comb(n, j) * math.comb(n - 2, j - 1) // n for j in range(1, n)]}')

### P4.6
**How to find it.** (a) Plane trees with $n$ vertices are the Łukasiewicz words of Proposition 3.13; enumerate the compositions of $n-1$ into $n$ parts and keep the valid ones. Depths come from the stack decoding. (b) The cycle-lemma sampler of Chapter 3, decoded iteratively. (c) A random BST is built by inserting a random permutation; its depths are computed while inserting.

In [ ]:
def depths(c):
    stack, d, out = [], {}, []
    for v, cv in enumerate(c):
        dv = d[stack.pop()] + 1 if stack else 0
        d[v] = dv; out.append(dv); stack.extend([v] * cv)
    return out
def valid(c):
    h = 1
    for j, x in enumerate(c):
        h += x - 1
        if h <= 0 and j < len(c) - 1: return False
    return h == 0
def compositions(total, parts):
    if parts == 1: yield (total,); return
    for first in range(total + 1):
        for rest in compositions(total - first, parts - 1): yield (first,) + rest
for n in range(1, 14):
    trees = [c for c in compositions(n - 1, n) if valid(c)] if n <= 11 else None
    if trees is None: break
    total = sum(sum(depths(c)) for c in trees)
    print(f'n={n:2}: {len(trees):6} trees, total path length {total:8}, formula {(4 ** (n - 1) - math.comb(2 * n - 2, n - 1)) // 2:8}')

In [ ]:
def random_tree(n, rng):
    w = [1] * (n - 1) + [0] * (n - 1); rng.shuffle(w)
    c, run = [], 0
    for b in w:
        if b: run += 1
        else: c.append(run); run = 0
    c.append(run)
    # the valid rotation starts right after the first minimum of the walk h (cycle lemma)
    h, best, arg = 0, 0, 0
    for i, x in enumerate(c):
        h += x - 1
        if h < best: best, arg = h, i + 1
    r = arg % n
    return c[r:] + c[:r]
rng = random.Random(11)
print('     n   mean depth   sqrt(pi n)/2   mean height   sqrt(pi n)   BST mean depth')
for n in [100, 1000, 10000, 100000]:
    reps = 40 if n <= 10000 else 8
    md_, hs = [], []
    for _ in range(reps):
        c = random_tree(n, rng); assert valid(c)
        d = depths(c); md_.append(sum(d) / n); hs.append(max(d))
    bst = []
    for _ in range(10):
        perm = list(range(min(n, 20000))); rng.shuffle(perm)
        root, left, right, dep = perm[0], {}, {}, {perm[0]: 0}
        for x in perm[1:]:
            y = root; dd = 1
            while True:
                side = left if x < y else right
                if y in side: y = side[y]; dd += 1
                else: side[y] = x; dep[x] = dd; break
        bst.append(np.mean(list(dep.values())))
    print(f'{n:6} {np.mean(md_):12.2f} {math.sqrt(math.pi * n) / 2:14.2f} {np.mean(hs):13.2f} {math.sqrt(math.pi * n):12.2f} {np.mean(bst):16.2f}')

The averages match $\frac12\sqrt{\pi n}$ closely; the heights approach $\sqrt{\pi n}$ more slowly (the correction is of order $1$, and the height fluctuates on the scale $\sqrt n$). The binary search trees (ten of them, with at most $20\,000$ keys) have mean depth $2(1+\frac1n)H_n-4\approx2\ln n-2.85$, tiny in comparison. The sampler uses the cycle lemma without trying all rotations: the valid rotation starts right after the *first* minimum of the walk $h_j-1=\sum_{i\le j}(c_i-1)$: before it the walk stays strictly above the minimum, after it never below, which is the proof of Theorem 3.11 read backwards.

In [ ]:
def stirling1_row(n):   # unsigned Stirling numbers of the first kind, from u(u+1)...(u+n-1)
    row = [1]
    for m in range(n):
        new = [0] * (len(row) + 1)
        for k, x in enumerate(row):
            new[k] += m * x; new[k + 1] += x
        row = new
    return row
for n in [5, 10, 50]:
    row = stirling1_row(n); tot = math.factorial(n)
    mean = Fraction(sum(k * x for k, x in enumerate(row)), tot)
    var = Fraction(sum(k * k * x for k, x in enumerate(row)), tot) - mean ** 2
    H = sum(Fraction(1, k) for k in range(1, n + 1)); H2 = sum(Fraction(1, k * k) for k in range(1, n + 1))
    print(f'n={n}: mean {float(mean):.6f} = H_n {float(H):.6f};  variance {float(var):.6f} = H_n - H_n^(2) {float(H - H2):.6f}')
def cycles(perm):
    seen, c = set(), 0
    for i in range(len(perm)):
        if i not in seen:
            c += 1
            while i not in seen: seen.add(i); i = perm[i]
    return c
n = 1000; samples = []
for _ in range(4000):
    p = list(range(n)); rng.shuffle(p); samples.append(cycles(p))
H = sum(1 / k for k in range(1, n + 1)); H2 = sum(1 / k ** 2 for k in range(1, n + 1))
print(f'n=1000, simulation: mean {np.mean(samples):.3f} (H_n = {H:.3f}), variance {np.var(samples):.3f} (H_n - H_n^(2) = {H - H2:.3f})')
plt.figure(figsize=(6, 3))
vals = Counter(samples); ks = sorted(vals)
plt.bar(ks, [vals[k] / len(samples) for k in ks], color='C0', alpha=0.7, label='simulation')
xs = np.linspace(min(ks), max(ks), 200)
plt.plot(xs, np.exp(-(xs - H) ** 2 / (2 * (H - H2))) / math.sqrt(2 * math.pi * (H - H2)), 'C1', label='normal (Goncharov)')
plt.xlabel('number of cycles'); plt.legend(); plt.tight_layout(); plt.show()